# SVM Lab: Iris Species Classification
Iris has **three** classes, so the ROC step uses a one-vs-rest (micro-average) approach.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

## Step 1: Load and Preprocess the Data

In [ ]:
# Load the Iris dataset (if your lab notebook already has a load cell, keep that one instead)
iris = load_iris(as_frame=True)
data = iris.frame
target = 'target'
sepal_col = 'sepal length (cm)'

In [ ]:
# View the first few rows
data.head()

In [ ]:
# Check for missing values
data.isnull().sum()

In [ ]:
# Visualize the relationship between the target (Iris type) and sepal length
plt.figure(figsize=(8, 5))
sns.boxplot(x=target, y=sepal_col, data=data)
plt.xticks([0, 1, 2], ['Setosa', 'Versicolor', 'Virginica'])
plt.title('Sepal length by Iris species')
plt.show()

In [ ]:
# Encode target variable as categorical
data[target] = data[target].astype('category')

# Separate features and target variable
X = data.drop(columns=[target])
y = data[target]

# Standardize numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the Data (80%/20%, random state 42)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Shapes
X_train.shape, X_test.shape, y_train.shape, y_test.shape

## Step 2: Train the Model

In [ ]:
# Initialize and train the SVC model (probability=True enables predict_proba)
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

# Predictions and predicted probabilities
y_pred = svm_model.predict(X_test)
y_pred_proba = svm_model.predict_proba(X_test)

# Return y_pred
y_pred

## Step 3: Confusion Matrix and Classification Report

In [ ]:
# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix

In [ ]:
# Classification report
print(classification_report(y_test, y_pred, target_names=['Setosa', 'Versicolor', 'Virginica']))

## Step 4: Best Parameters with Grid Search

In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

grid_search = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Best parameters
best_params = grid_search.best_params_
best_params

## Step 5: Prediction and Metrics

In [ ]:
# Train the optimized model
optimized_model = grid_search.best_estimator_
optimized_model.fit(X_train, y_train)

# Predicted probabilities for every class
y_pred_proba = optimized_model.predict_proba(X_test)

# One-vs-rest labels so ROC can be computed for a 3-class problem
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

# Micro-average ROC curve: fpr, tpr and thresholds
fpr, tpr, thresholds = roc_curve(y_test_bin.ravel(), y_pred_proba.ravel())

# AUC
roc_auc = auc(fpr, tpr)
roc_auc

In [ ]:
# Plot the ROC curve (micro-average plus one curve per species)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=3, label=f'Micro-average (AUC = {roc_auc:.3f})')
for i, name in enumerate(['Setosa', 'Versicolor', 'Virginica']):
    f, t, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
    plt.plot(f, t, linestyle='--', label=f'{name} (AUC = {auc(f, t):.3f})')
plt.plot([0, 1], [0, 1], 'k:', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()